# Notebook 3.0: BERTopic Modeling & HDBSCAN Hyperparameter Sweep (2018–2022)

**Research Question 2 (RQ2) — Career Topic Evolution Across Regimes**

This notebook:
1. Loads sampled submissions from `data/samples/2.0/` across all 7 subreddits (`2018-01-01` to `2022-12-31`).
2. Extracts text via `'text' if 'text' in df_subs.columns else 'title'`, applies `apply_topic_cleaning()`, and draws a balanced stratified subset (`topic_sample_per_subreddit` per subreddit).
3. Computes or loads cached `all-MiniLM-L6-v2` embeddings (`submissions_7subs_{len(df_subs)}_seed{seed}`) with `local_files_only`.
4. Runs an HDBSCAN hyperparameter sweep and ranks configurations by `coherence_npmi` and `topic_diversity`.
5. Fits the best BERTopic model, applies `reduce_outliers_safe()` when raw outlier rate > 20%, and exports models, tables (`data/final/tables/`), and figures (`data/final/figures/`).

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
from umap import UMAP

from src.modeling import set_seed
from src.modeling.load import draw_stratified_subset, load_samples
from src.modeling.text import apply_topic_cleaning
from src.modeling.cache import (
    load_embeddings,
    save_embeddings,
    save_parquet_cache,
    save_topic_model,
)
from src.modeling.topics import (
    evaluate_topics,
    fit_topic_model,
    reduce_outliers_safe,
    run_hdbscan_sweep,
    topic_name_map,
)
from src.modeling.tracking import log_experiment

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

seed = int(cfg['modeling']['seed'])
set_seed(seed)

samples_dir = project_root / cfg['paths']['data_samples_dir'] / '2.0'
cache_dir = project_root / cfg['modeling']['cache_dir']
exp_dir = project_root / cfg['modeling']['experiment_log_dir']
final_dir = project_root / cfg['paths']['final_dir']
tables_dir = final_dir / 'tables'
fig_dir = final_dir / 'figures'
for d in (cache_dir, exp_dir, tables_dir, fig_dir):
    d.mkdir(parents=True, exist_ok=True)

print(f'Topic modeling config loaded (seed={seed}, cache_dir={cache_dir}).')

## Step 1: Load Submissions, Clean Text, & Draw Stratified Subset
Use `text` (falling back to `title` if absent), clean with `apply_topic_cleaning()`, and balance across subreddits and quarters.

In [ ]:
df_subs = load_samples(
    samples_dir=samples_dir,
    subreddits=cfg['analysis']['subreddits'],
    dataset_types=['submissions'],
)

start_ts = pd.Timestamp(cfg['analysis']['start_date'], tz='UTC')
end_ts = pd.Timestamp(cfg['analysis']['end_date'], tz='UTC') + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
df_subs = (
    df_subs[(df_subs['created_utc_dt'] >= start_ts) & (df_subs['created_utc_dt'] <= end_ts)]
    .copy()
    .reset_index(drop=True)
)

text_col = 'text' if 'text' in df_subs.columns else 'title'
df_subs['cleaned_text'] = apply_topic_cleaning(df_subs[text_col])
df_subs = df_subs[df_subs['cleaned_text'].str.split().str.len() >= 3].reset_index(drop=True)

per_sub = int(cfg['modeling']['topic_sample_per_subreddit'])
df_subs = draw_stratified_subset(df_subs, n_per_subreddit=per_sub, seed=seed)
docs = df_subs['cleaned_text'].tolist()

print(f'Topic modeling corpus: {len(df_subs):,} stratified submissions across {df_subs["subreddit"].nunique()} subreddits.')
print(df_subs['subreddit'].value_counts())

## Step 2: Compute or Load Cached Sentence Embeddings
Precompute dense normalized embeddings once using `all-MiniLM-L6-v2` with `local_files_only` and cache to `data/processed/cache/`.

In [ ]:
embed_key = f'submissions_7subs_{len(df_subs)}_seed{seed}'
embeddings = load_embeddings(embed_key, cache_dir=cache_dir)

if embeddings is None:
    local_only = bool(cfg['modeling'].get('local_files_only', True))
    print(f'Computing embeddings with {cfg["modeling"]["embedding_model"]} (local_files_only={local_only})...')
    embedder = SentenceTransformer(
        cfg['modeling']['embedding_model'],
        local_files_only=local_only,
    )
    embeddings = embedder.encode(
        docs,
        show_progress_bar=True,
        normalize_embeddings=True,
        batch_size=64,
    )
    save_embeddings(embeddings, embed_key, cache_dir=cache_dir)
else:
    print(f'Loaded cached embeddings ({embed_key}) with shape: {embeddings.shape}')

## Step 3: HDBSCAN Hyperparameter Sweep
Sweep over `min_cluster_size` and `min_samples`, log experiments, and sort results by `coherence_npmi` and `topic_diversity`.

In [ ]:
umap_params = cfg['modeling']['umap']
hdbscan_base = {
    'metric': cfg['modeling']['hdbscan']['metric'],
    'cluster_selection_method': cfg['modeling']['hdbscan']['cluster_selection_method'],
    'prediction_data': bool(cfg['modeling']['hdbscan']['prediction_data']),
}

sweep_results = run_hdbscan_sweep(
    docs=docs,
    embeddings=embeddings,
    umap_params=umap_params,
    hdbscan_base=hdbscan_base,
    min_cluster_sizes=[25, 40, 60, 80],
    min_samples_list=[5, 10],
    umap_n_neighbors_list=[15],
    top_n_words=int(cfg['modeling']['top_n_words']),
    seed=seed,
    experiment_log_dir=exp_dir,
)

sweep_csv = tables_dir / 'topic_sweep_results.csv'
sweep_results.to_csv(sweep_csv, index=False)
print(f'Sweep results saved to: {sweep_csv}')

sorted_sweep = sweep_results.sort_values(
    by=['coherence_npmi', 'topic_diversity', 'outlier_pct'],
    ascending=[False, False, True],
)
print('--- Sweep Results Summary (sorted by NPMI & Topic Diversity) ---')
print(sorted_sweep[['umap_n_neighbors', 'min_cluster_size', 'min_samples', 'n_topics', 'outlier_pct', 'coherence_npmi', 'coherence_umass', 'topic_diversity']].to_string(index=False))

## Step 4: Fit Optimal Model & Apply Conditional Outlier Reduction
Select the top configuration (`10 <= n_topics <= 45` preferred, `outlier_pct < 0.25` when available, sorted by `coherence_npmi` and `topic_diversity`). If the raw outlier rate exceeds 20%, apply `reduce_outliers_safe()`.

In [ ]:
candidates = sweep_results[(sweep_results['n_topics'] >= 10) & (sweep_results['n_topics'] <= 45)]
if candidates.empty:
    candidates = sweep_results[sweep_results['n_topics'] >= 5]
if candidates.empty:
    candidates = sweep_results
low_out = candidates[candidates['outlier_pct'] < 0.25]
pool = low_out if not low_out.empty else candidates

best_row = pool.sort_values(
    by=['coherence_npmi', 'topic_diversity', 'outlier_pct'],
    ascending=[False, False, True],
).iloc[0]

best_nn = int(best_row['umap_n_neighbors'])
best_mcs = int(best_row['min_cluster_size'])
best_ms = int(best_row['min_samples'])
print(f'Selected best raw config: nn={best_nn}, min_cluster_size={best_mcs}, min_samples={best_ms}')

u_best = {**umap_params, 'n_neighbors': best_nn}
h_best = {**hdbscan_base, 'min_cluster_size': best_mcs, 'min_samples': best_ms}
topic_model, raw_topics = fit_topic_model(
    docs=docs,
    embeddings=embeddings,
    umap_params=u_best,
    hdbscan_params=h_best,
    top_n_words=int(cfg['modeling']['top_n_words']),
    seed=seed,
)
raw_metrics = evaluate_topics(docs, topic_model, raw_topics, top_n=int(cfg['modeling']['top_n_words']))
print('Raw metrics:', raw_metrics)

final_topics = raw_topics
final_metrics = raw_metrics
if raw_metrics['outlier_pct'] > 0.20 and raw_metrics['n_topics'] > 1:
    print(f'Raw outlier rate {raw_metrics["outlier_pct"]:.1%} > 20%; applying reduce_outliers_safe()...')
    topic_model, final_topics = reduce_outliers_safe(topic_model, docs, raw_topics, embeddings=embeddings)
    final_metrics = evaluate_topics(docs, topic_model, final_topics, top_n=int(cfg['modeling']['top_n_words']))
    log_experiment(
        params={'stage': 'post_reduce_outliers', 'min_cluster_size': best_mcs, 'min_samples': best_ms, 'seed': seed},
        metrics=final_metrics,
        output_dir=exp_dir,
    )
    print('Post-reduction metrics:', final_metrics)

name_map = topic_name_map(topic_model)
df_subs['topic_raw'] = raw_topics
df_subs['topic'] = final_topics
df_subs['topic_name'] = df_subs['topic'].map(name_map)
df_subs['quarter'] = (
    pd.to_datetime(df_subs['created_utc_dt'], utc=True)
    .dt.tz_convert(None)
    .dt.to_period('Q')
    .astype(str)
)

save_topic_model(topic_model, key='final_submissions_model', cache_dir=cache_dir)
save_parquet_cache(df_subs, key='topics_submissions_df', cache_dir=cache_dir)

topic_info = topic_model.get_topic_info().copy()
topic_info['label'] = topic_info['Topic'].map(name_map)
topic_info['top_words'] = topic_info['Topic'].apply(
    lambda tid: ', '.join(w for w, _ in (topic_model.get_topic(int(tid)) or [])[:8])
    if int(tid) != -1
    else 'OUTLIER'
)
topic_info.to_csv(tables_dir / 'topics_summary.csv', index=False)

sub_topic_ct = pd.crosstab(df_subs['subreddit'], df_subs['topic_name'], normalize='index') * 100
sub_topic_ct.round(2).to_csv(tables_dir / 'topic_by_subreddit.csv')

print('\nTop-10 Discovered Topics:')
print(topic_info[topic_info['Topic'] != -1].head(10)[['Topic', 'Count', 'label', 'top_words']].to_string(index=False))

## Step 5: Visualize Topic Clusters & Quarterly Topic Evolution
Save 2D UMAP projection and quarterly topic evolution plots to `data/final/figures/`.

In [ ]:
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.1, metric='cosine', random_state=seed)
coords_2d = umap_2d.fit_transform(embeddings)
df_subs['x'] = coords_2d[:, 0]
df_subs['y'] = coords_2d[:, 1]

df_plot = df_subs[df_subs['topic'] != -1]
top_topics = df_plot['topic'].value_counts().head(8).index

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df_plot[df_plot['topic'].isin(top_topics)],
    x='x',
    y='y',
    hue='topic_name',
    palette='tab10',
    alpha=0.65,
    s=14,
    linewidth=0,
)
plt.title(f'2D UMAP Projection of Top-8 Discovered Career Topics (N={len(df_subs):,})', fontsize=13, pad=12)
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Discovered Topics', fontsize=8.5)
plt.tight_layout()

scatter_path = fig_dir / 'topic_scatter.png'
plt.savefig(scatter_path, dpi=300)
plt.show()
print(f'Topic scatter plot saved to: {scatter_path}')

# Quarterly evolution of top-6 non-outlier topics
top6_names = [name_map[t] for t in df_plot['topic'].value_counts().head(6).index]
q_topic = df_subs.groupby(['quarter', 'topic_name']).size().unstack(fill_value=0)
q_share = q_topic.div(q_topic.sum(axis=1), axis=0) * 100
q_share_top = q_share[[c for c in top6_names if c in q_share.columns]]

plt.figure(figsize=(13, 5.5))
for col in q_share_top.columns:
    plt.plot(q_share_top.index, q_share_top[col], marker='o', markersize=3.5, linewidth=1.7, label=col)
plt.xticks(rotation=45, ha='right')
plt.title('Quarterly Evolution of Top-6 Submission Topics (2018Q1–2022Q4)', fontsize=13, pad=12)
plt.xlabel('Quarter')
plt.ylabel('Share of Submissions (%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8.5)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

evo_path = fig_dir / 'topic_evolution.png'
plt.savefig(evo_path, dpi=300)
plt.show()
print(f'Topic evolution plot saved to: {evo_path}')